# Phase 8 - End-to-End Pipeline Orchestration

This notebook runs or resumes Phases 1 through 7 through the canonical `run_pipeline.py` entry point.

In [ ]:
# Cell 1 - Locate the project root and import notebook dependencies.
from pathlib import Path
import json
import sys

import pandas as pd

project_root = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / 'configs' / 'config.yaml').exists()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

config_path = project_root / 'configs' / 'config.yaml'
print(f'Project root: {project_root}')

In [ ]:
# Cell 2 - Configure a resumable all-scenario pipeline run.
from run_pipeline import configure_logging, run_pipeline

SCENARIO = 'all'
SKIP_PREPROCESSING = True
FORCE_PHASES = set()
DEVICE = 'auto'

configure_logging('INFO')
print(f'Scenario: {SCENARIO}')
print(f'Skip preprocessing: {SKIP_PREPROCESSING}')
print(f'Forced phases: {sorted(FORCE_PHASES) or "none"}')

In [ ]:
# Cell 3 - Run the canonical Phase 1-7 pipeline or reuse validated artifacts.
pipeline_report = run_pipeline(
    config_path=config_path,
    scenario=SCENARIO,
    skip_preprocessing=SKIP_PREPROCESSING,
    force_phases=FORCE_PHASES,
    device=DEVICE,
)

print(f"Pipeline report: {pipeline_report['report_path']}")

In [ ]:
# Cell 4 - Display execution status and duration for every pipeline phase.
phase_summary = pd.DataFrame(pipeline_report['phases'])
phase_summary

In [ ]:
# Cell 5 - Validate successful completion and the required all-scenario outputs.
assert pipeline_report['success'] is True
assert phase_summary['phase'].tolist() == list(range(1, 8))
assert phase_summary['status'].isin({'executed', 'reused', 'validated'}).all()

summary_path = Path(pipeline_report['evaluation_artifacts']['summary_path'])
analysis_path = Path(pipeline_report['evaluation_artifacts']['analysis_markdown_path'])
assert summary_path.exists()
assert analysis_path.exists()

summary_comparison = pd.read_csv(summary_path)
assert summary_comparison.shape[0] == 4
summary_comparison

In [ ]:
# Cell 6 - Inspect the persisted orchestration metadata for reproducibility.
with open(pipeline_report['report_path'], 'r', encoding='utf-8') as file:
    persisted_report = json.load(file)

{
    'started_at': persisted_report['started_at'],
    'completed_at': persisted_report['completed_at'],
    'total_duration_seconds': persisted_report['total_duration_seconds'],
    'target_scenarios': persisted_report['target_scenarios'],
    'effective_force_phases': persisted_report['effective_force_phases'],
}